# 🚀 Whitebox QA EKS Mentor - Fine-Tuning Notebook

This notebook trains a custom **Qwen 2.5 / Qwen 3 (QLoRA)** model specialized in Whitebox QA EKS log analysis, root cause diagnosis, and defect report generation.

### Instructions:
1. **Runtime -> Change runtime type** -> Select **T4 GPU**.
2. Run all cells in order (`Ctrl + F9` or click play on each cell).

In [ ]:
# Step 1: Verify GPU is allocated
!nvidia-smi

In [ ]:
# Step 2: Install Unsloth and Training Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 3: Embed the QA Training Dataset (ChatML Format)
import json

dataset_examples = [
  {
    "messages": [
      {"role": "system", "content": "You are a Whitebox QA defect analysis analyst. Given EKS logs and run facts, write a concise, strictly factual defect report with exact citations."},
      {"role": "user", "content": "EVIDENCE:\nTOOL: crosshair\nRUN ID: bbbd08e3-c9ea-40e8-a68e-519f44d9f156\nEKS LOG:\n2026-10-27T18:52:35.109530+00:00 cell-001-wb-orchestrator activities.planning: planning.compatibility_excluded\nTool: crosshair 0.0.60 | Verdict: INCOMPATIBLE | Reason: CUSTOMER_CODE_INCOMPATIBLE_WITH_EXECUTION_ENV\nDetail: customer code version is unresolved; a provider that executes customer code may not run against an unproven environment\nPOD STATE: All pods running with 0 restarts."},
      {"role": "assistant", "content": "# crosshair - Not Triggered (Run bbbd08e3)\n**Verdict:** Planning defect   **Priority:** Medium   **Confidence:** High\n\n## 1. Summary\nwb-orchestrator excluded crosshair 0.0.60 during planning because the customer code version was unresolved and could not be verified against available execution environments.\n\n## 2. Run Metadata\n- Run ID: bbbd08e3-c9ea-40e8-a68e-519f44d9f156\n- Tool: crosshair (0.0.60)\n- Tenant: d606c2bf-6761-4df3-b33c-00f32026c165\n\n## 3. Evidence from QA EKS\n- `2026-10-27T18:52:35.109530+00:00` `activities.planning` `planning.compatibility_excluded` (cell-001-wb-orchestrator)\n    - crosshair 0.0.60: INCOMPATIBLE / CUSTOMER_CODE_INCOMPATIBLE_WITH_EXECUTION_ENV\n    - detail: customer code version is unresolved; a provider that executes customer code may not run against an unproven environment\n\n## 4. Root Cause\nwb-orchestrator excluded crosshair 0.0.60 during planning: verdict INCOMPATIBLE, reason CUSTOMER_CODE_INCOMPATIBLE_WITH_EXECUTION_ENV.\n\n## 5. Recommended Fix\n- **Owner:** Platform (wb-orchestrator environment resolution)\n- **Action:** Fix the customer code version resolution in the orchestrator or pin exact versions in repo lockfiles.\n\n## 6. QA Acceptance Criteria\n- On re-run, a worker task is created for crosshair.\n- S3 receives a valid crosshair output JSON.\n- Taxonomy gate changes from FAIL to PASS."}
    ]
  },
  {
    "messages": [
      {"role": "system", "content": "You are a Whitebox QA defect analysis analyst. Given EKS logs and run facts, write a concise, strictly factual defect report with exact citations."},
      {"role": "user", "content": "EVIDENCE:\nTOOL: crosshair\nRUN ID: 86a8ebb0-b28e-411e-82f2-74b467dfdf63\nEKS LOG:\nNo orchestrator planning events found for crosshair in the last 72 hours.\nEarliest log on cluster: 2026-10-27T15:27:34+00:00.\nPOD STATE: All pods running with 0 restarts."},
      {"role": "assistant", "content": "# crosshair - Not Triggered (Run 86a8ebb0)\n**Verdict:** Insufficient evidence   **Priority:** Low   **Confidence:** Low\n\n## 1. Summary\nNo log lines for this run or tool were found in the retained QA EKS logs. The root cause cannot be determined without archived logs.\n\n## 2. Run Metadata\n- Run ID: 86a8ebb0-b28e-411e-82f2-74b467dfdf63\n- Tool: crosshair\n\n## 3. Evidence from QA EKS\n- No decisive events found in retained log window.\n- Pod health: all pods Running with 0 restarts.\n\n## 4. Root Cause\nNo log line for this run was found. The cause cannot be determined from active QA EKS logs.\n\n## 5. Recommended Fix\n- **Owner:** Unknown\n- **Action:** Re-run the pipeline and extract logs immediately, or check S3/CloudWatch log archives.\n\n## 6. QA Acceptance Criteria\n- Fresh logs captured within retention window showing planning decision."}
    ]
  }
]

with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for item in dataset_examples:
        f.write(json.dumps(item) + "\n")

print(f"Created dataset.jsonl with {len(dataset_examples)} training pairs!")

In [ ]:
# Step 4: Load Base Model in 4-bit QLoRA
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Step 5: Format Dataset with ChatML
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def format_prompts(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# Step 6: Train the Model (QLoRA)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=50,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("Starting training loop...")
trainer.train()
print("Training completed successfully!")

In [ ]:
# Step 7: Export to GGUF Format for Ollama
model.save_pretrained_gguf(
    "qa-eks-mentor-trained",
    tokenizer,
    quantization_method="q4_k_m"
)

print("Model exported to GGUF format!")

In [ ]:
# Step 8: Download GGUF to Local Computer
from google.colab import files
import os

# Find the generated .gguf file
gguf_files = [f for f in os.listdir(".") if f.endswith(".gguf")]
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("Look inside qa-eks-mentor-trained/ directory for the .gguf file!")